# [9665] FastText 1

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 04/01/25 20:20:19


### Import libraries

In [ ]:
import numpy as np
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import FastText
import logging

In [ ]:
%%time

import spacy

2025-04-01 20:20:32.702875: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CPU times: user 9.12 s, sys: 1.77 s, total: 10.9 s
Wall time: 14 s


In [ ]:
# Load spaCy for lemmatization
nlp = spacy.load('en_core_web_sm')

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/vj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/vj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Setup loggimg
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

### Load data

In [ ]:
data = {
    'title': [
        "The Lion King", "Pirates of the Caribbean", "Frozen", "The Avengers",
        "Finding Nemo", "The Jungle Book", "Toy Story", "The Matrix",
        "Jurassic Park", "Star Wars", "Harry Potter and the Sorcerer's Stone",
        "Cinderella", "Beauty and the Beast", "Aladdin", "Moana",
        "The Incredibles", "Mulan", "Tangled", "Zootopia", "WALL-E",
        "Coco", "Ratatouille", "Big Hero 6", "The Princess and the Frog",
        "Inside Out", "Up", "Brave", "Luca", "Encanto", "Shrek"
    ],
    'description': [
        "A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        "Captain Jack Sparrow embarks on an adventure to find the legendary treasure of the Caribbean.",
        "A young princess with magical ice powers must save her kingdom from eternal winter.",
        "A team of superheroes comes together to fight a powerful enemy threatening the world.",
        "A young fish embarks on a journey across the ocean to find his father.",
        "A young boy befriends a wild bear and learns the ways of the jungle.",
        "A boy and his toys come to life and have various adventures in the real world.",
        "A hacker discovers the true nature of the virtual reality world he’s been living in.",
        "Scientists clone dinosaurs, leading to thrilling and dangerous consequences.",
        "A group of rebels band together to fight against an evil empire in a galaxy far, far away.",
        "A young wizard discovers his magical abilities and begins his training at Hogwarts.",
        "A kind young woman attends a royal ball with the help of her fairy godmother.",
        "A cursed prince and a village girl form an unlikely bond in an enchanted castle.",
        "A street rat discovers a magical lamp and unleashes a powerful genie.",
        "A determined young Polynesian girl sets out on a journey to restore balance to the world.",
        "A family of superheroes must hide their powers while facing a new villainous threat.",
        "A young Chinese woman disguises herself as a man to take her father’s place in the army.",
        "A lost princess with magical hair escapes from her tower and explores the world.",
        "A rabbit police officer and a sly fox work together to solve a major crime in Zootopia.",
        "A lonely robot embarks on a space journey to save Earth’s future.",
        "A young boy discovers his passion for music in the Land of the Dead.",
        "A rat dreams of becoming a chef in a famous Paris restaurant.",
        "A young robotics prodigy and his inflatable robot team up to fight crime.",
        "A young waitress kisses a frog and gets caught in a magical adventure.",
        "A young girl navigates the emotions inside her mind.",
        "A grumpy old man and a boy scout travel to South America in a floating house.",
        "A Scottish princess must undo a terrible curse.",
        "A young sea monster explores the human world with his best friend.",
        "A magical Colombian family must protect their home from destruction.",
        "An ogre embarks on a quest to rescue a princess and regain his swamp."
    ],
    'director': [
        "Jon Favreau", "Gore Verbinski", "Chris Buck & Jennifer Lee", "Joss Whedon",
        "Andrew Stanton", "Jon Favreau", "John Lasseter", "The Wachowskis",
        "Steven Spielberg", "George Lucas", "Chris Columbus", "Kenneth Branagh",
        "Bill Condon", "Guy Ritchie", "Ron Clements & John Musker", "Brad Bird",
        "Tony Bancroft & Barry Cook", "Nathan Greno & Byron Howard", "Byron Howard & Rich Moore",
        "Andrew Stanton", "Lee Unkrich", "Brad Bird", "Don Hall & Chris Williams",
        "Ron Clements & John Musker", "Pete Docter", "Pete Docter & Bob Peterson",
        "Mark Andrews & Brenda Chapman", "Enrico Casarosa", "Byron Howard & Jared Bush",
        "Andrew Adamson"
    ],
    'cast': [
        "Matthew Broderick, James Earl Jones", "Johnny Depp, Orlando Bloom",
        "Idina Menzel, Kristen Bell", "Robert Downey Jr., Chris Evans",
        "Albert Brooks, Ellen DeGeneres", "Bill Murray, Ben Kingsley",
        "Tom Hanks, Tim Allen", "Keanu Reeves, Laurence Fishburne",
        "Sam Neill, Laura Dern", "Mark Hamill, Harrison Ford",
        "Daniel Radcliffe, Emma Watson", "Lily James, Richard Madden",
        "Emma Watson, Dan Stevens", "Will Smith, Naomi Scott",
        "Auli'i Cravalho, Dwayne Johnson", "Craig T. Nelson, Holly Hunter",
        "Ming-Na Wen, Eddie Murphy", "Mandy Moore, Zachary Levi",
        "Ginnifer Goodwin, Jason Bateman", "Ben Burtt, Elissa Knight",
        "Anthony Gonzalez, Gael García Bernal", "Patton Oswalt, Ian Holm",
        "Ryan Potter, Scott Adsit", "Anika Noni Rose, Bruno Campos",
        "Amy Poehler, Phyllis Smith", "Edward Asner, Jordan Nagai",
        "Kelly Macdonald, Emma Thompson", "Jacob Tremblay, Jack Dylan Grazer",
        "Stephanie Beatriz, John Leguizamo", "Mike Myers, Eddie Murphy"
    ]
}

In [ ]:
dataset = pd.DataFrame(data)

In [ ]:
#df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/DisneyPlus_titles.csv')
dataset.shape

(30, 4)

### Prepare data

In [ ]:
# Clean up any NaN values in the dataset
dataset = dataset.fillna('')

### Preprocess text

In [ ]:
# Preprocessing: Tokenization, Stopword Removal, Lemmatization, and Punctuation Removal
def preprocess(text):
    # Check if text is a valid string
    if isinstance(text, str):
        # Remove all punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Tokenize the text
        tokens = word_tokenize(text.lower())

        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

        # Lemmatization using spaCy
        lemmatized_tokens = [token.lemma_ for token in nlp(" ".join(tokens)).doc]

        return lemmatized_tokens
    else:
        return []  # Return an empty list if the text is not a valid string

In [ ]:
# Concatenate title, description, director, and cast to create a combined string for each entry
dataset['combined'] = dataset['title'] + " " + dataset['description'] + " " + dataset['director'] + " " + dataset['cast']

In [ ]:
# Prepare the text data by applying preprocessing to the 'combined' column
processed_titles = [preprocess(text) for text in dataset['combined']]

In [ ]:
for item in processed_titles[0:3]:
    print(item)

['lion', 'king', 'young', 'lion', 'prince', 'flee', 'kingdom', 'learn', 'true', 'meaning', 'responsibility', 'bravery', 'jon', 'favreau', 'matthew', 'broderick', 'james', 'earl', 'jones']
['pirate', 'caribbean', 'captain', 'jack', 'sparrow', 'embarks', 'adventure', 'find', 'legendary', 'treasure', 'caribbean', 'gore', 'verbinski', 'johnny', 'depp', 'orlando', 'bloom']
['frozen', 'young', 'princess', 'magical', 'ice', 'power', 'must', 'save', 'kingdom', 'eternal', 'winter', 'chris', 'buck', 'jennifer', 'lee', 'idina', 'menzel', 'kristen', 'bell']


### Train FastText model with cleaned data

In [ ]:
# Train the FastText model with the preprocessed data
model = FastText(sentences=processed_titles, vector_size=100, window=5, min_count=1, epochs=50)

2025-04-01 20:20:37,693 : INFO : collecting all words and their counts
2025-04-01 20:20:37,694 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2025-04-01 20:20:37,695 : INFO : collected 383 word types from a corpus of 498 raw words and 30 sentences
2025-04-01 20:20:37,696 : INFO : Creating a fresh vocabulary
2025-04-01 20:20:37,698 : INFO : FastText lifecycle event {'msg': 'effective_min_count=1 retains 383 unique words (100.00% of original 383, drops 0)', 'datetime': '2025-04-01T20:20:37.698627', 'gensim': '4.3.2', 'python': '3.8.8 (default, Apr 13 2021, 12:59:45) \n[Clang 10.0.0 ]', 'platform': 'macOS-10.16-x86_64-i386-64bit', 'event': 'prepare_vocab'}
2025-04-01 20:20:37,699 : INFO : FastText lifecycle event {'msg': 'effective_min_count=1 leaves 498 word corpus (100.00% of original 498, drops 0)', 'datetime': '2025-04-01T20:20:37.699517', 'gensim': '4.3.2', 'python': '3.8.8 (default, Apr 13 2021, 12:59:45) \n[Clang 10.0.0 ]', 'platform': 'macOS-10.16-x86_6

In [ ]:
# Create function to find most similar Disney+ titles based on a query
def get_most_similar(query, model, topn=5):
    query_tokens = preprocess(query)

    if not query_tokens:
        return []

    # Compute query vector
    query_vector = np.mean([model.wv[word] for word in query_tokens if word in model.wv], axis=0)

    # Get most similar words
    similar_words = model.wv.similar_by_vector(query_vector, topn=topn * 2)  # Retrieve more to avoid duplicates

    # Store titles with highest similarity scores
    title_similarities = {}

    for word, similarity in similar_words:
        matching_rows = dataset[dataset['combined'].str.contains(word, case=False, na=False)]

        for _, row in matching_rows.iterrows():
            title = row['title']
            if title not in title_similarities or similarity > title_similarities[title]:
                title_similarities[title] = similarity  # Keep the highest similarity score

    # Sort and return top results
    return sorted(title_similarities.items(), key=lambda x: x[1], reverse=True)[:topn]

In [ ]:
# Example queries
queries = [
    "magical adventure with a princess",  # Matches fairytale/fantasy movies
    "pirate treasure hunt",  # Matches adventure movies
    "robot saving the world",  # Matches sci-fi/robotic themes
    "superheroes fighting villains",  # Matches superhero movies
    "boy lost in the jungle",  # Matches jungle-themed survival movies
    "wizard learning magic",  # Matches fantasy/magic-based movies
    "aliens invading Earth",  # Matches sci-fi alien movies
    "animals talking in a city",  # Matches animated animal adventure movies
    "time travel adventure",  # Matches sci-fi/time travel themes
    "kids on a mystery-solving adventure",  # Matches detective/kids adventure movies
    "musical with great songs",  # Matches musicals
    "young girl discovering her true power",  # Matches coming-of-age/magic themes
    "scientist making a big discovery",  # Matches science/discovery-themed movies
    "epic space battle",  # Matches space opera/sci-fi movies
    "historical battle for a kingdom",  # Matches historical/fantasy war movies
    "a chef with a big dream",  # Matches food-based movies
    "lonely robot exploring space",  # Matches AI/robot-themed movies
    "a cursed prince in an enchanted castle",  # Matches fantasy fairytales
    "a young warrior defends their homeland",  # Matches action/martial arts movies
    "comedy about an unusual friendship",  # Matches buddy comedy movies
    "a sports team trying to win the championship",  # Matches sports drama movies
    "an adventurer searching for lost treasure",  # Matches Indiana Jones-style adventure movies
    "a musician trying to follow their dreams",  # Matches music-themed movies
]

In [ ]:
# Display the results for each query
i = 0
for query in queries:
    i += 1
    print(f"Query {i}: {query}")
    similar_titles = get_most_similar(query, model)  # Function retrieves similar movie titles

    print(f"Top 5 Similar Titles:")
    for title, similarity in similar_titles:
        print(f"\t{title}: Similarity: {similarity:.2f}")

    print("-" * 40)

Query 1: magical adventure with a princess
Top 5 Similar Titles:
	Pirates of the Caribbean: Similarity: 0.99
	Toy Story: Similarity: 0.99
	The Princess and the Frog: Similarity: 0.99
	Frozen: Similarity: 0.99
	Harry Potter and the Sorcerer's Stone: Similarity: 0.99
----------------------------------------
Query 2: pirate treasure hunt
Top 5 Similar Titles:
	Pirates of the Caribbean: Similarity: 0.98
	The Incredibles: Similarity: 0.97
	Moana: Similarity: 0.97
	Tangled: Similarity: 0.97
	Luca: Similarity: 0.97
----------------------------------------
Query 3: robot saving the world
Top 5 Similar Titles:
	The Avengers: Similarity: 0.99
	Toy Story: Similarity: 0.99
	The Matrix: Similarity: 0.99
	Moana: Similarity: 0.99
	Tangled: Similarity: 0.99
----------------------------------------
Query 4: superheroes fighting villains
Top 5 Similar Titles:
	The Incredibles: Similarity: 0.99
	Finding Nemo: Similarity: 0.99
	Mulan: Similarity: 0.99
	The Matrix: Similarity: 0.99
	Harry Potter and the So